### Load Benchmark Problems

In [1]:
sampling_for_lite = False # True for MineCEraft Lite
sampling_rand_seed = 42

In [2]:
# --- Load prompts from the mapping (single source of truth) ---
from pathlib import Path
import json
import random

# Load JSON files from benchmarks folder only (not archive/; archive = excluded from evaluation)
# Schema: prompts = [[turn1, turn2, ...], ...], checks = [[eval_turn1], [eval_turn2], ...]
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
json_files = sorted(BENCHMARKS_DIR.glob("*.json"))
mapping = []
for json_file in json_files:
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Build runs: each run = (prompt_sequence, checks_per_turn, _comment)
runs = []
if sampling_for_lite:
    rng = random.Random(sampling_rand_seed)
    for json_file in json_files:
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        file_runs = []
        for item in file_mapping:
            for prompt_sequence in item["prompts"]:
                file_runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))
        if file_runs:
            runs.append(rng.choice(file_runs))
else:
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))

print(f"[PY] Benchmarks dir: {BENCHMARKS_DIR.resolve()} (cwd: {Path.cwd().resolve()})")
if len(runs) == 0:
    print("[PY] ⚠️ No runs. Put at least one .json in benchmarks/ (not in archive/), or run notebook from the folder that contains benchmarks/.")
print("Total problem #:", len(runs))
for i, (prompts, _, _) in enumerate(runs):
    print(i, prompts)

[PY] Benchmarks dir: D:\git\mineCEraft\mineCEraft\benchmarks (cwd: D:\git\mineCEraft\mineCEraft)
Total problem #: 6
0 ["Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step."]
1 ["Install the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step."]
2 ["Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think about the order in which we should place the blocks to optimize the movement path."]
3 ["Install the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think about the order in which we should place the blocks to optimize the movement path."]
4 ['Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep.']
5 ['Install the foundation for a 6x6 block rectangular building. Use stone blocks and make it 

### Builder Agent Run

In [3]:
# Add Agent
from pathlib import Path
import subprocess
import time

parent_dir = Path.cwd().parent
proc_main_agent = subprocess.Popen(["node", "main.js"], cwd=str(parent_dir))
print(f"Running (PID={proc_main_agent.pid})")

time.sleep(15) # wait until agent is added to the world

# Send Prompts
import subprocess, shutil, json
from pathlib import Path
from action_processor import read_placed_and_removed_from_action

# Path to Node script in current directory
script = (Path.cwd() / "send_prompts.js").resolve()
node = shutil.which("node") or "node"

proc_send_prompts = subprocess.Popen(
    [node, str(script)],
    cwd=str(script.parent),       # Node's process.cwd() equals the JS folder
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,                    # line-buffered
)

# Build payload from runs (Node still expects flat prompts + run_lengths)
prompts_for_send = [p for (seq, _, _) in runs for p in seq]
run_lengths_for_send = [len(seq) for (seq, _, _) in runs]
proc_send_prompts.stdin.write(json.dumps({
    "prompts": prompts_for_send,
    "run_lengths": run_lengths_for_send,
    "clear_between": True,
    "inter_prompt_command": "Come up to the highest block position, and move 20 blocks in the positive z direction.",
    "inter_prompt_delay": 10000
}) + "\n")
proc_send_prompts.stdin.close()

SENTINEL = "::ACTION_MAX_JS::" # indicator of the lastly executed JS file names for each prompt.
# Each entry: (placed, removed) for that turn (for multi-turn we merge into cumulative coords in evaluation)
arr_of_deltas = []

try:
    for line in proc_send_prompts.stdout:
        line = line.rstrip("\n")
        print(line)  # always mirror Node logs

        # If Node reports the max-numbered file, parse and print it
        if line.startswith(SENTINEL):
            payload_raw = line[len(SENTINEL):]
            try:
                payload = json.loads(payload_raw)
            except json.JSONDecodeError:
                print("[PY] Failed to parse sentinel JSON.")
                continue

            if payload.get("ok") and "path" in payload:
                file_path = Path(payload["path"])
                print(f"\n[PY] Max action file: index={payload.get('index')} name={payload.get('name')}")
                print(f"[PY] Path: {file_path}")

                try:
                    placed, removed = read_placed_and_removed_from_action(str(file_path))
                    print(f"[PY] placed={len(placed)}, removed={len(removed)}")
                    arr_of_deltas.append((placed, removed))
                                        
                except Exception as e:
                    print(f"[PY] Failed to convert action to coords: {e}")
                    
            else:
                # e.g., dir not found or no js files
                reason = payload.get("reason", "unknown")
                print(f"[PY] No max file reported (reason={reason}).")
finally:
    proc_send_prompts.stdout.close()
    proc_send_prompts.wait(timeout=10) 

    proc_main_agent.kill()
    print("Process terminated.")


Running (PID=44616)
ℹ️ Using agent name: builder
🧹 Cleared all files under D:\git\mineCEraft\bots\builder\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=1Fh_KpZqmdNAhIKAAAAD)

➡️ Sending to builder (run 1, turn 1/1): "Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step."
⏳ Waiting for completion keyword (timeout 10 min)...
📨 [builder] I'll start by laying the foundation.  First, I need to mark the corner of the foundation. !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder] <|header_end|> !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder]  !placeHere("stone")
📨 [builder] The foundation is marked. Next, I will create a 6x6 stone structure. !newAction("Build a 6x6 stone structure with a 3 block height")
📨 [builder] I've successfull

### Evaluation

In [4]:
import json
import csv
from datetime import datetime
import importlib
from pathlib import Path
from collections import defaultdict

# Filled when evaluation runs (matching arr_of_deltas); use in Visualization cell: coords_by_problem[problem_i][turn_j]
coords_by_problem = []

# Use runs from Load cell; only load mapping/runs if running this cell standalone (runs not defined)
try:
    runs
except NameError:
    BENCHMARKS_DIR = Path.cwd() / "benchmarks"
    mapping = []
    for json_file in sorted(BENCHMARKS_DIR.glob("*.json")):
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        mapping.extend(file_mapping)
    runs = []
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))

def resolve_callable(dotted: str):
    """Resolve e.g. 'size.is_equal' to eval_code.size.is_equal callable."""
    fq = f"eval_code.{dotted}"
    mod_name, func_name = fq.rsplit(".", 1)
    mod = importlib.import_module(mod_name)
    return getattr(mod, func_name), fq

def run_checks(coords, checks):
    """Run checks on coords; return score, results, and by_category summary.
    Each check must return a value in [0, 1]; we clamp and sum for score."""
    results = []
    score = 0.0
    cat_pass = defaultdict(float)   # category -> sum of scores in [0, 1]
    cat_total = defaultdict(int)  # category -> total count

    for chk in checks:
        fn_name = chk["fn"]              # e.g., 'material.is_quantity_correct'
        args = chk.get("args") or {}
        category = fn_name.split(".", 1)[0]  # module name as category (e.g, material)

        fn, fq = resolve_callable(fn_name)
        try:
            v = fn(coords, **args)
            ok = max(0.0, min(1.0, float(v)))
        except Exception as e:
            ok, args = 0.0, {**args, "_error": str(e)}  # surface error on this line

        score += ok
        cat_total[category] += 1
        cat_pass[category]  += ok

        results.append({"fn": fq, "category": category, "ok": ok, "args": args})

    # shape as plain dicts for printing
    cat_summary = {
        c: {"pass": cat_pass[c], "total": cat_total[c]}
        for c in sorted(cat_total.keys())
    }
    return {"score": score, "total": len(results), "results": results, "by_category": cat_summary}

overall_pass = 0.0
overall_total = 0
overall_by_cat_pass  = defaultdict(float)
overall_by_cat_total = defaultdict(int)

# Load category.json for top-level category summary
with open(Path("category.json"), encoding="utf-8") as _f:
    CATEGORIES = json.load(_f)

def merge_coords(prev: list, placed: list, removed: list) -> list:
    """Return prev with removed positions dropped and placed added (multi-turn cumulative)."""
    rem_set = {(c["x"], c["y"], c["z"]) for c in removed}
    prev_remaining = [c for c in prev if (c["x"], c["y"], c["z"]) not in rem_set]
    by_key = {(c["x"], c["y"], c["z"]): c for c in prev_remaining}
    for c in placed:
        by_key[(c["x"], c["y"], c["z"])] = c
    return list(by_key.values())

total_turns = sum(len(seq) for (seq, _, _) in runs)
if len(arr_of_deltas) != total_turns:
    print(f"[PY] Skipping evaluation: arr_of_deltas has {len(arr_of_deltas)} entries but runs have {total_turns} turns.")
    print("[PY] Run the construction cell (send prompts) first, wait for all prompts to complete, then run this evaluation cell.")
if len(arr_of_deltas) == total_turns:
    results_dir = Path("eval_results")
    results_dir.mkdir(exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    builder_cfg = json.loads((Path.cwd().parent / "builder.json").read_text(encoding="utf-8"))
    
    model_safe = builder_cfg["model"].replace("/", "-").replace(" ", "_")[:60]
    print(model_safe)
    
    log_path = results_dir / f"eval_{model_safe}_{ts}.log"
    csv_path = results_dir / f"eval_{model_safe}_{ts}.csv"
    log_file = open(log_path, "w", encoding="utf-8")
    csv_rows = []

    def print_and_log(msg):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    coord_index = 0
    coords_by_problem = []
    for run_idx, run_tuple in enumerate(runs):
        prompt_sequence, checks_per_turn = run_tuple[0], run_tuple[1]
        run_comment = run_tuple[2] if len(run_tuple) > 2 else ""
        n_turns = len(prompt_sequence)
        cumulative = []
        coords_this_problem = []
        for turn_idx in range(n_turns):
            placed, removed = arr_of_deltas[coord_index]
            coord_index += 1
            cumulative = merge_coords(cumulative, placed, removed)
            prompt_text = prompt_sequence[turn_idx]
            coords = cumulative
            checks_this_turn = checks_per_turn[turn_idx]
            
            report = run_checks(coords, checks_this_turn)
            coords_this_problem.append(cumulative)
            
            # --- Pretty print per-turn report (and log file) ---
            print_and_log("\n[PY] === Evaluation Result ===")
            print_and_log(f"[PY] Run #{run_idx+1}, Turn #{turn_idx+1}/{n_turns}: {prompt_text}")
            print_and_log(f"[PY] Score: {report['score']} / {report['total']} (coords={len(coords)})")

            # Category breakdown for this turn
            print_and_log("[PY] Category scores:")
            for cat, st in report["by_category"].items():
                print_and_log(f"  - {cat}: {st['pass']} / {st['total']}")

            # Individual checks (and collect rows for CSV)
            for r in report["results"]:
                ok_val = r["ok"]
                status = "PASS" if ok_val >= 1.0 else ("FAIL" if ok_val <= 0 else f"{ok_val:.2f}")
                line = f"  · {status} | {r['fn']}({r.get('args', {})})"
                print_and_log(line)
                csv_rows.append({
                    "run_idx": run_idx + 1,
                    "turn_idx": turn_idx + 1,
                    "prompt_text": prompt_text,
                    "check_fn": r["fn"],
                    "check_args": str(r.get("args", {})),
                    "passed": ok_val,
                    "_comment": run_comment,
                })
            
            # Accumulate overall totals
            overall_pass  += report["score"]
            overall_total += report["total"]
            for cat, st in report["by_category"].items():
                overall_by_cat_pass[cat]  += st["pass"]
                overall_by_cat_total[cat] += st["total"]
        coords_by_problem.append(coords_this_problem)

    # --- Overall summary across all prompts ---
    if overall_total > 0:
        print_and_log("\n[PY] === Overall Summary ===")
        overall_pct = (overall_pass / overall_total * 100) if overall_total > 0 else 0.0
        print_and_log(f"[PY] Total PASS: {overall_pass} / {overall_total} ({overall_pct:.1f}%)")
        # By top-level category (from category.json)
        print_and_log("[PY] By category:")
        for big_cat in CATEGORIES:
            p = sum(overall_by_cat_pass.get(sub, 0) for sub in CATEGORIES[big_cat])
            t = sum(overall_by_cat_total.get(sub, 0) for sub in CATEGORIES[big_cat])
            cat_pct = (p / t * 100) if t > 0 else 0.0
            print_and_log(f"  - {big_cat}: {p} / {t} ({cat_pct:.1f}%)")
        print_and_log("[PY] By subcategory:")
        for big_cat in CATEGORIES:
            for sub in CATEGORIES[big_cat]:
                p = overall_by_cat_pass.get(sub, 0)
                t = overall_by_cat_total.get(sub, 0)
                if t > 0:
                    cat_pct = (p / t * 100)
                    print_and_log(f"  - {sub}: {p} / {t} ({cat_pct:.1f}%)")

    # Save CSV (one row per check: filter by prompt_text or check_fn)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["run_idx", "turn_idx", "prompt_text", "check_fn", "check_args", "passed", "_comment"])
        writer.writeheader()
        writer.writerows(csv_rows)
    print_and_log(f"\n[PY] Results saved: log={log_path}, csv={csv_path}")
    log_file.close()

    if len(coords_by_problem) == 0:
        print_and_log("[PY] ⚠️ coords_by_problem is empty. Run Load → Construction → Evaluation (benchmarks/ must have .json files).")


meta-llama-llama-4-scout-17b-16e-instruct

[PY] === Evaluation Result ===
[PY] Run #1, Turn #1/1: Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step.
[PY] Score: 2.5238095238095237 / 4 (coords=144)
[PY] Category scores:
  - efficiency: 0.5238095238095238 / 1
  - material: 1.0 / 1
  - physical_plausibility: 1.0 / 1
  - size: 0.0 / 1
  · PASS | eval_code.material.is_all_material_equal_to({'expected_material': 'stone'})
  · FAIL | eval_code.size.is_equal({'xz': [6, 6], 'y': 1})
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · 0.52 | eval_code.efficiency.l1_dist({})

[PY] === Evaluation Result ===
[PY] Run #2, Turn #1/1: Install the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step.
[PY] Score: 2.5238095238095237 / 4 (coords=144)
[PY] Category scores:
  - efficiency: 0.5238095238095238 / 1
  - material: 1.0 / 1
  - physical_p

### Double-check & Visualization

In [5]:
# coords_by_problem[problem_index][turn_index] = coords at that eval (problem = load cell print index).
from eval_code.viz import plotly_blocks

problem_index = 0   # problem number from load cell (print(i, prompts))
turn_index = -1     # 0=first turn, -1=last turn for that problem

try:
    coords_by_problem
except NameError:
    print("Run the evaluation cell first so coords_by_problem is available.")
else:
    n = len(coords_by_problem)
    if n == 0:
        print("No problems in coords_by_problem. Run Load → Construction → Evaluation (with matching arr_of_deltas), then try again.")
    elif problem_index < 0 or problem_index >= n:
        print(f"problem_index must be 0..{n-1} (there are {n} problems). Current problem_index={problem_index}")
    else:
        turns = coords_by_problem[problem_index]
        if turns:
            plotly_blocks.plot(turns[turn_index])
        else:
            print(f"Problem {problem_index} has no turns/coords.")
            
runs[problem_index]

(["Lay the foundation for a 6x6 block rectangular building. Use stone blocks and make it one block deep. Let's think step by step."],
 [[{'fn': 'material.is_all_material_equal_to',
    'args': {'expected_material': 'stone'}},
   {'fn': 'size.is_equal', 'args': {'xz': [6, 6], 'y': 1}},
   {'fn': 'physical_plausibility.is_ground_connected', 'args': {}},
   {'fn': 'efficiency.l1_dist', 'args': {}}]],
 '')

In [6]:
coords_by_problem[0]

[[{'x': 9, 'y': 0, 'z': 349, 'material': 'stone'},
  {'x': 9, 'y': 0, 'z': 350, 'material': 'stone'},
  {'x': 9, 'y': 0, 'z': 351, 'material': 'stone'},
  {'x': 9, 'y': 0, 'z': 352, 'material': 'stone'},
  {'x': 9, 'y': 0, 'z': 353, 'material': 'stone'},
  {'x': 9, 'y': 0, 'z': 354, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 349, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 350, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 351, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 352, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 353, 'material': 'stone'},
  {'x': 10, 'y': 0, 'z': 354, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 349, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 350, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 351, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 352, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 353, 'material': 'stone'},
  {'x': 11, 'y': 0, 'z': 354, 'material': 'stone'},
  {'x': 12, 'y': 0, 'z': 349, 'material': 'stone'},
  {'x': 12, 'y': 0